In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [6]:
!pip install --no-cache-dir --upgrade "pyarrow==14.0.2" "datasets==2.14.6"



In [7]:
!pip install -q evaluate nltk bert-score
import nltk
nltk.download("wordnet")
nltk.download("omw-1.4")


[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [9]:
###############################################################
# 0) CLEAN ENVIRONMENT — NO Trainer, NO Accelerate, NO XLA
###############################################################
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

###############################################################
# 1) INSTALL DEPENDENCIES
###############################################################
print("Installing dependencies (nltk + bert-score)...")
!pip install -q nltk bert-score

import nltk
nltk.download("wordnet")
nltk.download("omw-1.4")
print("✓ Dependencies installed\n")


###############################################################
# 2) IMPORTS
###############################################################
import torch
from torch.utils.data import DataLoader

import numpy as np
import random
from datasets import load_dataset
from bert_score import score as bertscore_score
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq
)

from tqdm.auto import tqdm


###############################################################
# 3) CONFIG
###############################################################
MODEL_NAME = "t5-base"

MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 64

BATCH_SIZE = 4
ACCUM_STEPS = 4
LR = 2e-4
EPOCHS = 2

CHECKPOINT_EVERY = 5000       # save every 5k steps
CHECKPOINT_DIR = "/kaggle/working/checkpoints"   # temp but survives until reboot
FINAL_DIR = "/kaggle/output/t5-final"       # PERSISTENT

SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device, "\n")


###############################################################
# 4) LOAD DATA
###############################################################
print("Loading SQuAD v2 dataset...")
dataset = load_dataset("squad_v2")

train_split = dataset["train"].train_test_split(test_size=0.2, seed=SEED)
train_ds = train_split["train"]
val_ds   = train_split["test"]
test_ds  = dataset["validation"]

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}\n")


###############################################################
# 5) TOKENIZATION
###############################################################
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(batch):
    inputs = ["generate question: " + c for c in batch["context"]]
    
    enc = tokenizer(
        inputs,
        truncation=True,
        padding="max_length",
        max_length=MAX_SOURCE_LENGTH
    )
    
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch["question"],
            truncation=True,
            padding="max_length",
            max_length=MAX_TARGET_LENGTH
        )
    enc["labels"] = labels["input_ids"]
    return enc

print("Tokenizing...")
train_tok = train_ds.map(preprocess, batched=True)
val_tok   = val_ds.map(preprocess, batched=True)
test_tok  = test_ds.map(preprocess, batched=True)

train_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_tok.set_format(type="torch", columns=["input_ids", "attention_mask"])

collator = DataCollatorForSeq2Seq(tokenizer)

print("✓ Tokenization complete\n")


###############################################################
# 6) LOAD MODEL (with RESUME ability)
###############################################################
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Detect latest checkpoint
resume_path = None
checkpoints = sorted([d for d in os.listdir(CHECKPOINT_DIR)], reverse=True)
if checkpoints:
    resume_path = os.path.join(CHECKPOINT_DIR, checkpoints[0])
    print("Resuming from checkpoint:", resume_path)
else:
    print("No checkpoint found — starting fresh.")

if resume_path:
    model = AutoModelForSeq2SeqLM.from_pretrained(resume_path)
else:
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)


###############################################################
# 7) TRAINING LOOP WITH CHECKPOINTING
###############################################################
train_loader = DataLoader(train_tok, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator)

global_step = 0
print("\n🔥 Training Started...\n")

for epoch in range(EPOCHS):
    print(f"\n============================")
    print(f"      EPOCH {epoch+1}/{EPOCHS}")
    print("============================\n")

    model.train()
    total_loss = 0

    pbar = tqdm(train_loader, total=len(train_loader))
    optimizer.zero_grad()

    for batch in pbar:
        global_step += 1
        
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            out = model(**batch)
            loss = out.loss / ACCUM_STEPS

        loss.backward()
        total_loss += loss.item()

        if global_step % ACCUM_STEPS == 0:
            optimizer.step()
            optimizer.zero_grad()

        pbar.set_description(f"Loss {loss.item():.4f}")

        ###########################################
        # PERIODIC CHECKPOINT
        ###########################################
        if global_step % CHECKPOINT_EVERY == 0:
            ckpt_path = f"{CHECKPOINT_DIR}/step_{global_step}"
            os.makedirs(ckpt_path, exist_ok=True)
            model.save_pretrained(ckpt_path)
            tokenizer.save_pretrained(ckpt_path)
            print(f"\n💾 Saved checkpoint at step {global_step} to {ckpt_path}")

    print(f"Epoch {epoch+1} Avg Loss = {total_loss/len(train_loader):.4f}")

print("\n🎉 TRAINING COMPLETE!\n")


###############################################################
# 8) FINAL SAVE (PERSISTENT)
###############################################################
print("Saving final model to PERSISTENT Kaggle directory...")
os.makedirs(FINAL_DIR, exist_ok=True)
model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print("✓ Final model saved at:", FINAL_DIR)


Installing dependencies (nltk + bert-score)...
✓ Dependencies installed

Using device: cuda 

Loading SQuAD v2 dataset...


[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Train: 104255 | Val: 26064 | Test: 11873

Tokenizing...


Map:   0%|          | 0/104255 [00:00<?, ? examples/s]

Map:   0%|          | 0/26064 [00:00<?, ? examples/s]

...

✓ Tokenization complete

No checkpoint found — starting fresh.

🔥 Training Started...


      EPOCH 1/2



  0%|          | 0/26064 [00:00<?, ?it/s]


💾 Saved checkpoint at step 5000 to /kaggle/working/checkpoints/step_5000

💾 Saved checkpoint at step 10000 to /kaggle/working/checkpoints/step_10000

💾 Saved checkpoint at step 15000 to /kaggle/working/checkpoints/step_15000

💾 Saved checkpoint at step 20000 to /kaggle/working/checkpoints/step_20000

💾 Saved checkpoint at step 25000 to /kaggle/working/checkpoints/step_25000
Epoch 1 Avg Loss = 0.1088

      EPOCH 2/2



  0%|          | 0/26064 [00:00<?, ?it/s]


💾 Saved checkpoint at step 30000 to /kaggle/working/checkpoints/step_30000

💾 Saved checkpoint at step 35000 to /kaggle/working/checkpoints/step_35000

💾 Saved checkpoint at step 40000 to /kaggle/working/checkpoints/step_40000

💾 Saved checkpoint at step 45000 to /kaggle/working/checkpoints/step_45000

💾 Saved checkpoint at step 50000 to /kaggle/working/checkpoints/step_50000
Epoch 2 Avg Loss = 0.0927

🎉 TRAINING COMPLETE!

Saving final model to PERSISTENT Kaggle directory...
✓ Final model saved at: /kaggle/output/t5-final


In [10]:
import os

print("Does the directory exist?", os.path.exists("/kaggle/working/t5-final"))
print("\nContents of /kaggle/working/:")
print(os.listdir("/kaggle/working"))

print("\nIf the folder exists, list its files:")
if os.path.exists("/kaggle/working/t5-final"):
    print(os.listdir("/kaggle/working/t5-final"))


Does the directory exist? False

Contents of /kaggle/working/:
['.virtual_documents', 'checkpoints']

If the folder exists, list its files:


In [11]:
###############################################################
# 0) IMPORTS
###############################################################
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
from torch.utils.data import DataLoader

import numpy as np
from datasets import load_dataset
import evaluate
from bert_score import score as bertscore_score

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq
)

from tqdm.auto import tqdm


###############################################################
# 1) LOAD TRAINED MODEL (persistent)
###############################################################
MODEL_PATH = "/kaggle/output/t5-final"

print("Loading trained model from:", MODEL_PATH)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("✓ Model loaded.\n")


###############################################################
# 2) LOAD SQuAD TEST SPLIT
###############################################################
print("Loading SQuAD v2 test split...")
dataset = load_dataset("squad_v2")["validation"]

print("Test examples:", len(dataset), "\n")


###############################################################
# 3) TOKENIZE TEST SET
###############################################################
def preprocess_test(batch):
    inputs = ["generate question: " + c for c in batch["context"]]
    enc = tokenizer(
        inputs,
        padding="max_length",
        truncation=True,
        max_length=512
    )
    return enc

print("Tokenizing test set...")
test_tok = dataset.map(preprocess_test, batched=True)

test_tok.set_format(type="torch", columns=["input_ids", "attention_mask"])

collator = DataCollatorForSeq2Seq(tokenizer)

test_loader = DataLoader(test_tok, batch_size=8, shuffle=False)


###############################################################
# 4) GENERATE QUESTIONS
###############################################################
print("\n🔮 Generating questions for test set...")

preds = []
refs  = dataset["question"]

for batch in tqdm(test_loader):
    batch = {k: v.to(device) for k, v in batch.items()}

    with torch.no_grad():
        out = model.generate(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            max_length=64,
            num_beams=4
        )

    decoded = tokenizer.batch_decode(out, skip_special_tokens=True)
    preds.extend(decoded)

print("✓ Generation complete.\n")


###############################################################
# 5) COMPUTE METRICS
###############################################################
###############################################################
# 5) COMPUTE METRICS  (FIXED VERSION)
###############################################################
print("Calculating metrics...")

# Convert references to list (critical fix!)
refs = list(dataset["question"])
refs = refs[:len(preds)]   # ensure equal length

# ROUGE
rouge = evaluate.load("rouge")
rouge_out = rouge.compute(predictions=preds, references=refs)

# METEOR
meteor = evaluate.load("meteor")
meteor_out = meteor.compute(predictions=preds, references=refs)

# BERTScore
P, R, F1 = bertscore_score(
    preds,
    refs,
    lang="en",
    model_type="bert-base-uncased"
)

print("\n=========== FINAL TEST METRICS ===========")
print("ROUGE-1:", rouge_out["rouge1"])
print("ROUGE-2:", rouge_out["rouge2"])
print("ROUGE-L:", rouge_out["rougeL"])
print("METEOR :", meteor_out["meteor"])
print("BERTScore F1:", float(F1.mean()))
print("==========================================\n")


###############################################################
# 6) SHOW SAMPLE OUTPUTS
###############################################################
print("\n======== SAMPLE OUTPUTS ========\n")

for i in range(10):
    print(f"Sample {i+1}:")
    print("Context:", dataset[i]["context"][:300], "...")
    print("Actual :", refs[i])
    print("Pred   :", preds[i])
    print("--------------------------------\n")


Loading trained model from: /kaggle/output/t5-final
✓ Model loaded.

Loading SQuAD v2 test split...
Test examples: 11873 

Tokenizing test set...


Map:   0%|          | 0/11873 [00:00<?, ? examples/s]


🔮 Generating questions for test set...


  0%|          | 0/1485 [00:00<?, ?it/s]

✓ Generation complete.

Calculating metrics...


...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]


=========== FINAL TEST METRICS ===========
ROUGE-1: 0.2556375370486529
ROUGE-2: 0.08618114673936356
ROUGE-L: 0.23363258380910573
METEOR : 0.24266238170729715
BERTScore F1: 0.5735303163528442


======== SAMPLE OUTPUTS ========

Sample 1:
Context: The Normans (Norman: Nourmands; French: Normands; Latin: Normanni) were the people who in the 10th and 11th centuries gave their name to Normandy, a region in France. They were descended from Norse ("Norman" comes from "Norseman") raiders and pirates from Denmark, Iceland and Norway who, under their ...
Actual : In what country is Normandy located?
Pred   : What did the Normans give their name to in the 10th and 11th centuries?
--------------------------------

Sample 2:
Context: The Normans (Norman: Nourmands; French: Normands; Latin: Normanni) were the people who in the 10th and 11th centuries gave their name to Normandy, a region in France. They were descended from Norse ("Norman" comes from "Norseman") raiders and pirates from Denmark, Ice

In [12]:
import os

print("Does final dir exist?", os.path.exists("/kaggle/output/t5-final"))

print("\nFiles in /kaggle/output/:")
print(os.listdir("/kaggle/output/"))


Does final dir exist? True

Files in /kaggle/output/:
['t5-final']


In [13]:
import shutil

src = "/kaggle/output/t5-final"
dst = "/kaggle/working/t5-final"

print("Copying...")
shutil.copytree(src, dst)

print("Done. Check the left panel.")


Copying...
Done. Check the left panel.


In [15]:
import shutil

src = "/kaggle/output/t5-final"
dst = "/kaggle/working/t5-final-zipped"

print("Copying...")
shutil.copytree(src, dst)

print("Zipping...")
shutil.make_archive(dst, 'zip', dst)

print("Done. Check the left panel for the zipped file.")


Copying...
Zipping...
Done. Check the left panel for the zipped file.
